In [71]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from huggingface_hub import InferenceClient

from dotenv import load_dotenv
import os

In [72]:
load_dotenv()

api_key = os.getenv('HUGGINGFACE_API_KEY')
if not api_key:
    raise ValueError("HUGGINGFACE_API_KEY not found in .env file. Please add your API key.")
print(f"API key loaded: {api_key[:10]}...")


API key loaded: hf_IiqrVKZ...


In [73]:
llm = HuggingFaceEndpoint(
    repo_id='google/flan-t5-base',
    huggingfacehub_api_token=api_key
) 

In [74]:
model =ChatHuggingFace(llm=llm)

In [75]:
# define state
class LLMstate(TypedDict):
    question : str
    answer : str

In [76]:
# define graph
graph  = StateGraph(LLMstate)


In [77]:
def Llm_call_qa(state :LLMstate )-> LLMstate:
    # extract question from state 
     quest = state['question']
    # preapred prompt and send to model
     prompt = f'provide the appropriate answer for this {quest}'
    # extarct ans 
     answer = model.invoke(prompt).content
# set answer to state
     state['answer'] = answer
     return state

In [78]:
# define nodes 
if 'Llm_call_qa' not in graph.nodes:
    graph.add_node('Llm_call_qa', Llm_call_qa)

# define edge
graph.add_edge(START,'Llm_call_qa')
graph.add_edge('Llm_call_qa',END)

In [79]:
# compile graph
workflow= graph.compile()
initial_state = {'question':'what is kidlins law or problem solving'}
final_state = workflow.invoke(initial_state)
print(final_state)


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69bec4f4-0104d63034cc522974346236;8f01ce1f-cd85-4302-bf73-9a6aa5665c6c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.